# INF201 - Deliverable 2
## Campus Coffee Shop Sales Pipeline

**Group:** G6_2: Nikolai Friis, Tommy Rustad Apelseth, Aron Lerdahl 

**GitHub repository:** https://github.com/tommyra06/INF201-Deliverables 

---

### Contents
* Setup
* Task 1: File ingestion and regular expressions
* Task 2: Recursive utility functions
* Task 3: Cleaning and analysing data using NumPy
* Task 4: Data analysis using Pandas
* Task 5: Full workflow
* Use of generative AI

---

## Setup

### Virtual environment

A new virtual environment (`.venv`) was created for Deliverable 2, and the packages used are listed within `requirements.txt`. To recreate the environment: 

```bash
python -m venv .venv
pip install -r requirements.txt
```
### Input data 
The input data is generated by running the `generate_data_for_d2.py` which is in `Oppgavefiler/` from the project folder. `BASE_DIR` in the script is set to `Path("d2_data")`, so the data is saved in the `d2_data/` folder next to this notebook. The script creates these: 

* `d2_data/2026/<store>/<register>/register_sales.log` - log files in mixed encodings (UTF-8 and ISO-8859-1 (Latin-1))
* `d2_data/store_config.json` - nested store and register metadata

The `d2_data/` folder is not uploaded to GitHub, so the script must be run before running this notebook.

### Imports and file paths
All imports used in the notebook are collected in the cell below, together with the paths to the input data.

In [ ]:
# Library imports that come with Python
import json                                         # Reads store_config.json (Task 2) as well as save hourly_revenue.json (Task 5)  
import re                                           # Used for regular expressions to parse valid sales lines in the logs (Task 1)  
import pathlib                                      # File and folder paths: Reading files (Task 1) and search (Task 2)  

# Installed libraries, listed in `requirements.txt`
import numpy as np                                  # Array calculations, such as outlier cleaning and refund ratios (Task 3)  
import pandas as pd                                 # DataFrames, resampling, pivot tables, and saving CSV files (Task 4 and Task 5) 
import matplotlib.pyplot as plt                     # Plotting results (Task 5)
import chardet as cd                                # Detects encoding (Task 1)

# Paths to the input data
DATA_DIR: pathlib.Path = pathlib.Path("d2_data")                    # Folder with the generated log files and config (Task 2)
CONFIG_FILE: pathlib.Path = DATA_DIR / "store_config.json"  # Nested store and register metadata (Task 2)
### OBS, MIGHT BE USED FOR TASK 5, NOT SURE YET. IF NOT IN NEED DELETE THIS COMMENT
### IF IN NEED, DELETE IT AND CHANGE THE COMMENT FOR DATA_DIR AND CONFIG_FILE TO INCLUDE "(Task 2 and 5)" ABOVE 


Matplotlib is building the font cache; this may take a moment.


## Task 1: File ingestion and regular expressions







In [ ]:

def file_reader(path: str | pathlib.Path) -> tuple[str, str]:
    '''
    Detects file encoding and returns content as string.

    Parameters
    ----------
    path : str | pathlib.Path
        Directory to file. Converted to a pathlib.Path object.

    Returns
    -------
    tuple[str,str]
        First string is file content.
        Second string is file encoding
    '''
    file_dir = pathlib.Path(path)
    info = ''

    with open(file_dir, 'rb') as file:
        raw = file.read()

    result = cd.detect(raw)
    encoding = result['encoding']

    with open(file_dir, 'r', encoding=encoding) as f:
            data = f.readlines()
            for line in data:
                info += line
    return(info, encoding)



    



NameError: name 'pathlib' is not defined

In [ ]:

def log_parser(path: str) -> list[dict]:
    '''
    Recursively matches each line in file to pattern.

    Parameters
    ----------
        Directory to file
    
    Returns
    -------
        List of dictionaries for each line in the file.
    '''
    info = []
    pattern = re.compile(
            r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
            r"\[(?P<register_id>[^\]]+)\]\s+"
            r"item=(?P<item>\S+)\s+"
            r"price=(?P<price>\d+\.\d+)\s+"
            r"(?P<currency>[A-Z]{3})\s+"
            r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)"
        )
    data = file_reader(path)[0]
    data = data.split('\n')
    for line in data:
        print(line)
        match = pattern.match(line)
        if match:
            data = match.groupdict()
            data['price'] = float(data['price'])
            info.append(data)
    return(info)


---

## Task 2: Recursive utility functions

For Task 2, we are writing two recursive helper functions that are used in the full workflow in Task 5.

1. `find_log_files` - searches through the nested `d2_data/` folder structure (year → store → register) and returns all `.log` files.
2. `find_active_registers` - traverses the nested `store_config.json` file and returns the IDs of all registers marked as active.

Both functions use explicit recursion: each call handles one level of the structure and calls itself on the levels below. As required, we do not use `Path.glob()`, `Path.rglob()` or `os.walk()`.

---

### 2.1 Recursive search for log files

The function takes a folder path and converts it to a `pathlib.Path` object. It then loops over everything directly inside the folder using `Path.iterdir()`:

* If the item is a **folder**, the function calls itself on that folder and adds the files it ends up finding to the list (this is the recursive step).
* If the item is a **file** with the right extension, it is added to the list.
* Other files (e.g. `store_config.json`) are ignored.

The recursion stops when a folder only contains files, since no new calls are made, which is the base case. The list is sorted and returned, so the order of files is the same every time the code runs.

The input path has the type hint `str | Path`, since the function accepts both a string and a `Path` object.

In [ ]:
def find_log_files(path: str | pathlib.Path, extension: str = ".log") -> list[pathlib.Path]:
    """
    Recursively find all files with a given extension in a directory tree.    
    
    Parameters                                                               
    ----------
    path : str | Path
        The directory to search in. Converted to a pathlib.Path object.
    extension : str, optional
        The file extension to search for, including the dot (default ".log").
        
    Returns                                                                       
    -------
    list[Path]
        Sorted list of Path objects for all matching files in the directory 
        and all of its subdirectories.
    """
    path = pathlib.Path(path)
    found: list[pathlib.Path] = []

    for item in path.iterdir():
        if item.is_dir():
            # Recursive step: Search inside the subdirectory 
            found.extend(find_log_files(item, extension))
        elif item.is_file() and item.suffix == extension: 
            found.append(item)

    return sorted(found) 

---

### 2.2 Recursive search for active registers

The configuration file contains dictionaries and lists nested inside each other. Since we don't want the function to depend on the exact structure, it handles any combination of the two:

* If the input is a **dictionary**, we check whether it has `"active": True` and an `"id"`. If so, the id is added to the set. The function then calls itself on every value in the dictionary.
* If the input is a **list**, the function calls itself on every element.
* Anything else (strings, numbers, booleans, `None`) returns an empty set. This is the base case.

The results from each recursive call are combined using set union (`|=`). We return a `set[str]` so the same register id is never stored twice. Note that `true` in JSON becomes `True` in Python when the file is loaded with `json.load`.

To give the input a precise type hint, we define the type alias `JSONValue`, which describes every type a value loaded from a JSON file can have. The alias refers to itself, since a dictionary or list can contain more dictionaries and lists.

In [3]:
JSONValue = dict[str, "JSONValue"] | list["JSONValue"] | str | int | float | bool | None


def find_active_registers(node: JSONValue) -> set[str]:
    """
    Recursively collect the ids of all active registers in a nested JSON structure.

    Parameters
    ----------
    node : JSONValue
        Any part of the loaded JSON data: a dictionary, a list or a single value.

    Returns
    -------
    set[str]
        The ids of all registers where "active" is True. Returns an empty set
        if node is not a dictionary or a list (the base case).
    """
    active: set[str] = set()

    if isinstance(node, dict):
        if node.get("active") is True and "id" in node:
            active.add(str(node["id"]))
        # Recursive step: search every value in the dictionary
        for value in node.values():
            active |= find_active_registers(value)

    elif isinstance(node, list):
        # Recursive step: search every element in the list
        for element in node:
            active |= find_active_registers(element)

    return active

---

### Testing each function 

We test both functions on the data generated by `generate_data_for_d2.py`.

In [ ]:
log_files: list[pathlib.Path] = find_log_files(DATA_DIR)
print(f"Found {len(log_files)} log files:")
for file in log_files:
    print(" ", file)

with open(CONFIG_FILE, encoding="utf-8") as f:
    config: JSONValue = json.load(f)

active_registers: set[str] = find_active_registers(config)
print("\nActive registers:", sorted(active_registers))

FileNotFoundError: [Errno 2] No such file or directory: 'd2_data'

---

### Results and discussion of Task 2

`find_log_files` found all 9 log files, one for each register (`POS_01`-`POS_03`) in each of the three stores.

`find_active_registers` returned `{'POS_01', 'POS_02', 'POS_03'}`. `POS_04` is not included since it is marked as `"active": false` in the config.

One thing to note is that the config file only lists register ids, not which store each register belongs to. All registers that appear in the log files are active, so filtering the transactions by active register in Task 5 will **not** remove any rows with this dataset.

---

# Use of Generative AI

### Task 1
Use of Gen AI:


---


### Task 2 

**Questions about the code**
* I asked what each part of the two functions does. Claude explained how `find_log_files` converts the input to a `Path`, uses `iterdir()` to look at one folder at a time, and calls itself on subfolders, and how `find_active_registers` checks each dictionary for `"active": True` and calls itself on every value in dictionaries and lists.
* I asked what was meant by the *base case* in `find_active_registers`. It is when the input is a single value (string, number, bool or `None`), so there is nothing more to search and an empty set is returned. This is what stops the recursion.
* After my first version of `find_log_files` only found one file, I asked where `return sorted(found)` should be placed. It had been indented inside the `for` loop, so the function returned after the first item. It must line up with the loop so the whole folder is searched first.
* I asked why the docstrings were so long, and learned the difference between NumPy-style and Google-style docstrings. We kept NumPy style, with `Parameters` and `Returns` sections.
* I asked what `DATA_DIR` was and where it came from. It is a constant defined in the Setup cell, a relative path to the `d2_data/` folder, written in capitals by convention.

**Questions about the structure**
* How a notebook with many cells is run, and whether the imports at the top are available in the cells below. Variables and functions stay in memory after a cell has run, so the Setup cell only needs to run once, first.
* Why each import is needed and which task uses it, which I wrote as a comment next to each import.
* Why the data path had to match our generator, which saves to `d2_data/` instead of `data/`.

**Errors and corrections**
Claude helped me find errors in my notebook: `extedn` instead of `extend`, `return` indented inside the loop, and `node-values()` instead of `node.values()`, as well as spelling mistakes in the docstrings and markdown.

**What I did myself**
I put together the notebook, adapted the paths and text to our project, fixed the errors and ran the code on our generated data to check that it found all 9 log files and the active registers `{'POS_01', 'POS_02', 'POS_03'}`.

---

### Task 3
Use of Gen AI:

---

### Task 4
Use of Gen AI:

---

### Task 5
Use of Gen AI:

---